### Langkah 1 - Melanjutkan alur project

Bagian ini menjalankan langkah berikutnya dalam proses kerja notebook. Bacalah bersama output di bawahnya, karena hasil dari cell ini biasanya menjadi jembatan menuju tahap berikutnya.

In [ ]:
# https://www.kaggle.com/competitions/commonlitreadabilityprize# memprediksi kemudahan pembacaan suatu bagian

### Langkah 2 - Menyiapkan alat kerja

Kita mulai dengan memanggil library yang akan dipakai sepanjang notebook. Dengan menaruhnya di awal, pembaca bisa langsung melihat alat apa saja yang dibutuhkan sebelum project dijalankan.

In [ ]:
import typing as t
from ast import literal_eval

from transformer.models.regressor import RegressorLM
from transformer.dataloaders.inference import InferenceDataModule
from transformer.params.transformer import TransformerParams

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from lightning import Trainer
from lightning.pytorch.callbacks.early_stopping import EarlyStopping
from transformers import LlamaTokenizer

### Langkah 3 - Memuat dan mengecek data awal

Di sini data dimuat lalu langsung ditampilkan beberapa baris pertamanya. Cara ini membantu pembaca memastikan file terbaca dengan benar sebelum masuk ke proses analisis.

In [ ]:
# memuat dan melihat pratinjau datadata = pd.read_csv("data/commonlit.csv")
# mengecualikan:# - kutipan di luar [-3, 1] karena tidak dapat diandalkan# - outlier tunggal dengan kesalahan standar peringkat kemudahan noldata = data.loc[
    data["BT Easiness"].between(-3, 1) & (data["BT s.e."] > 0), 
    ["Excerpt", "BT Easiness", "BT s.e."]
]
# mengonversi baris baru menjadi spasidata.Excerpt = data.Excerpt.str.replace("\n", " ")
data.head()

### Langkah 4 - Membaca pola lewat visualisasi

Visualisasi dipakai untuk melihat pola yang sulit ditangkap dari angka mentah. Dari grafik ini, pembaca bisa lebih mudah memahami hubungan antar kolom sebelum masuk ke tahap model.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 3))
data["BT Easiness"].plot.hist(ax=axs[0], title="Easiness Rating")
data["BT s.e."].plot.hist(ax=axs[1], title="Std. Error")
plt.tight_layout()
plt.show()

### Langkah 5 - Melanjutkan alur project

Bagian ini menjalankan langkah berikutnya dalam proses kerja notebook. Bacalah bersama output di bawahnya, karena hasil dari cell ini biasanya menjadi jembatan menuju tahap berikutnya.

In [ ]:
# mengonversi teks menjadi daftar stringX = data["Excerpt"].to_list()

### Langkah 6 - Menyamakan skala fitur

Bagian ini membuat nilai fitur berada pada skala yang lebih seimbang. Ini membantu model belajar lebih adil, terutama saat beberapa kolom memiliki rentang angka yang jauh berbeda.

In [ ]:
# gunakan skor keterbacaan berskala min-maks sebagai keluaranscaler = MinMaxScaler()
scaler.fit(np.array([[-3.0], [1.0]]))
y = torch.from_numpy(scaler.transform(data[["BT Easiness"]])).float()

### Langkah 7 - Melanjutkan alur project

Bagian ini menjalankan langkah berikutnya dalam proses kerja notebook. Bacalah bersama output di bawahnya, karena hasil dari cell ini biasanya menjadi jembatan menuju tahap berikutnya.

In [ ]:
# gunakan kebalikan dari kesalahan standar sebagai bobot sampel fungsi kerugianweights = torch.from_numpy(data[["BT s.e."]].pow(-1).to_numpy()).float()

### Langkah 8 - Membuat fungsi bantu

Bagian ini membungkus proses tertentu ke dalam fungsi atau class agar kode berikutnya tidak berulang. Dengan cara ini, notebook menjadi lebih rapi dan alurnya lebih mudah dirawat.

In [ ]:
# membuat modul dataclass CommonlitReadabilityDataModule(InferenceDataModule):
    def setup(self: t.Self, stage: str) -> None:
        self.X, self.y, self.weights = X, y, weights
        super().setup(stage=stage)

### Langkah 9 - Menyiapkan teks untuk model

Data teks perlu dibersihkan dan diubah menjadi bentuk numerik sebelum bisa dipelajari model. Langkah ini membantu kata-kata penting tetap terbaca sebagai pola, bukan sekadar kalimat mentah.

In [ ]:
# inisialisasi tokenizer yang telah dilatih sebelumnya# - llama tidak menambahkan token EOS secara default, jadi timpa ini# - llama juga tidak menggunakan token padding, jadi ini perlu ditambahkantokenizer = LlamaTokenizer.from_pretrained(
    "huggyllama/llama-7b", add_eos_token=True, legacy=False
)
tokenizer.add_special_tokens({"pad_token": "<pad>"})

### Langkah 10 - Menyiapkan teks untuk model

Data teks perlu dibersihkan dan diubah menjadi bentuk numerik sebelum bisa dipelajari model. Langkah ini membantu kata-kata penting tetap terbaca sebagai pola, bukan sekadar kalimat mentah.

In [ ]:
# melihat distribusi panjang urutan tokendata["Excerpt"].apply(tokenizer.tokenize).str.len().plot.hist(bins=50)

### Langkah 11 - Menyiapkan teks untuk model

Data teks perlu dibersihkan dan diubah menjadi bentuk numerik sebelum bisa dipelajari model. Langkah ini membantu kata-kata penting tetap terbaca sebagai pola, bukan sekadar kalimat mentah.

In [ ]:
# inisialisasi trafocontext_length = 300
model = RegressorLM(
    params=TransformerParams(context_length=context_length),
    tokenizer=tokenizer,
)

### Langkah 12 - Menyiapkan teks untuk model

Data teks perlu dibersihkan dan diubah menjadi bentuk numerik sebelum bisa dipelajari model. Langkah ini membantu kata-kata penting tetap terbaca sebagai pola, bukan sekadar kalimat mentah.

In [ ]:
# memberi token & mengkodekan data dan menyiapkan pemisahan pelatihan/pengujiandatamodule = CommonlitReadabilityDataModule(
    tokenizer=tokenizer,
    context_length=context_length,
    batch_size=32,
    val_size=0.2,
    test_size=0.1,
    num_workers=9,
    persistent_workers=True,
    limit=None,
    random_state=1,
)

### Langkah 13 - Melatih model

Bagian ini menjalankan proses training agar model belajar dari data yang sudah disiapkan. Output dari langkah ini biasanya menjadi petunjuk awal apakah model mulai menangkap pola dengan baik.

In [ ]:
%%time
# melatih modelnyatrainer = Trainer(
    max_epochs=50,
    callbacks=EarlyStopping(monitor="val_loss", mode="min", patience=5),
    accelerator="cpu",
)
trainer.fit(model=model, datamodule=datamodule)

### Langkah 14 - Membuat prediksi

Di sini model digunakan untuk menghasilkan prediksi dari data yang tersedia. Langkah ini menunjukkan bagaimana model dipakai setelah selesai dilatih, sehingga alurnya terasa lengkap dari data sampai hasil.

In [ ]:
# melihat kumpulan prediksi set pengujian pertamapred = trainer.predict(model=model, datamodule=datamodule)
pred[:10]

### Langkah 15 - Melanjutkan alur project

Bagian ini menjalankan langkah berikutnya dalam proses kerja notebook. Bacalah bersama output di bawahnya, karena hasil dari cell ini biasanya menjadi jembatan menuju tahap berikutnya.

In [ ]:
# menghitung akurasitorch.tensor([x[1] == x[2] for batch in pred for x in batch]).float().mean()